# Experiment: WOP vs WOS (C++) on Unit Cube

Objective:
- Compare `wop` and `wos` on the exterior Dirichlet problem for the unit cube `[0, 1]^3` from the same start point.
- Run only C++ computations (Python is used only to orchestrate build and subprocess calls).
- Evaluate quality and cost for `n = 100, 1000, 10000, 100000, 1000000`.
            


In [1]:
from __future__ import annotations

import json
import math
import subprocess
import time
from pathlib import Path
from pprint import pprint

SEED_BASE = 314159
N_VALUES = [100, 1000, 10000, 100000, 1000000]
METHODS = ["wop", "wos"]
X0 = (1.5, 0.5, 0.5)
MAX_STEPS = 1_000_000

DELTA = 1e-3
RHO_SCALE = 1.0
RHO1_SCALE = 2.0

R_MAX = 1e6
R_MAX_MODE = "project"
R_MAX_FACTOR = 3.0


def find_workspace_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "external_wop_cpp").exists():
            return candidate
    raise FileNotFoundError("Could not find workspace root with external_wop_cpp")


WORKSPACE_ROOT = find_workspace_root(Path.cwd())
CPP_ROOT = WORKSPACE_ROOT / "external_wop_cpp"
TMP_ROOT = WORKSPACE_ROOT / "tmp" / "jupyter-notebook" / "wop-vs-wos-cpp-unit-cube"
TMP_ROOT.mkdir(parents=True, exist_ok=True)

print("WORKSPACE_ROOT =", WORKSPACE_ROOT)
print("CPP_ROOT       =", CPP_ROOT)
print("TMP_ROOT       =", TMP_ROOT)
            


WORKSPACE_ROOT = c:\Users\Master\Desktop\Data\Диплом
CPP_ROOT       = c:\Users\Master\Desktop\Data\Диплом\external_wop_cpp
TMP_ROOT       = c:\Users\Master\Desktop\Data\Диплом\tmp\jupyter-notebook\wop-vs-wos-cpp-unit-cube


## Plan

- Build a small C++ runner that can call `estimate_wop` and `estimate_wos_box` on the same unit-cube setup.
- Execute both methods for each `n` from `N_VALUES` with fixed seeds.
- Collect `J`, `abs_error`, `eps`, `mean_steps`, and wall-clock runtime.

Reference harmonic function: `u(x) = 1 / ||x - a||` with `a = (0.25, 0.35, 0.45)` (inside cube).
The boundary value is `f(y) = u(y)` on cube faces, so exact value at `x0` is known.
            


In [2]:
runner_cpp = TMP_ROOT / "main.cpp"
runner_cmake = TMP_ROOT / "CMakeLists.txt"

runner_cpp.write_text(r'''
#include <cmath>
#include <cstdint>
#include <exception>
#include <iomanip>
#include <iostream>
#include <optional>
#include <sstream>
#include <stdexcept>
#include <string>

#include "wop/geometry/box.hpp"
#include "wop/solver/wop_solver.hpp"
#include "wop/solver/wos_box_solver.hpp"

namespace {

struct CliArgs {
    std::string method = "wop";
    wop::math::Vec3 x0{1.5, 0.5, 0.5};
    int n = 1000;
    std::uint64_t seed = 12345;
    int max_steps = 1'000'000;
    bool json = false;

    double delta = 1e-3;
    double rho_scale = 1.0;
    double rho1_scale = 2.0;

    std::optional<double> r_max = 1e6;
    wop::solver::RMaxMode r_max_mode = wop::solver::RMaxMode::Project;
    double r_max_factor = 3.0;
};

wop::math::Vec3 parse_vec3_arg(const std::string& text) {
    std::istringstream iss(text);
    double x = 0.0;
    double y = 0.0;
    double z = 0.0;
    std::string extra;
    if (!(iss >> x >> y >> z) || (iss >> extra)) {
        throw std::invalid_argument("Invalid --x0 format, expected: \"1.5 0.5 0.5\".");
    }
    return wop::math::Vec3{x, y, z};
}

wop::solver::RMaxMode parse_r_max_mode(const std::string& text) {
    if (text == "escape") {
        return wop::solver::RMaxMode::Escape;
    }
    if (text == "project") {
        return wop::solver::RMaxMode::Project;
    }
    throw std::invalid_argument("Invalid --r-max-mode, expected: escape|project.");
}

CliArgs parse_args(int argc, char** argv) {
    CliArgs args;
    for (int i = 1; i < argc; ++i) {
        const std::string key = argv[i];
        auto require_value = [&](const std::string& option_name) -> std::string {
            if (i + 1 >= argc) {
                throw std::invalid_argument("Missing value for " + option_name + ".");
            }
            return argv[++i];
        };

        if (key == "--method") {
            args.method = require_value("--method");
        } else if (key == "--x0") {
            args.x0 = parse_vec3_arg(require_value("--x0"));
        } else if (key == "--n") {
            args.n = std::stoi(require_value("--n"));
        } else if (key == "--seed") {
            args.seed = static_cast<std::uint64_t>(std::stoull(require_value("--seed")));
        } else if (key == "--max-steps") {
            args.max_steps = std::stoi(require_value("--max-steps"));
        } else if (key == "--delta") {
            args.delta = std::stod(require_value("--delta"));
        } else if (key == "--rho-scale") {
            args.rho_scale = std::stod(require_value("--rho-scale"));
        } else if (key == "--rho1-scale") {
            args.rho1_scale = std::stod(require_value("--rho1-scale"));
        } else if (key == "--r-max") {
            const double value = std::stod(require_value("--r-max"));
            args.r_max = (value > 0.0) ? std::optional<double>(value) : std::nullopt;
        } else if (key == "--r-max-mode") {
            args.r_max_mode = parse_r_max_mode(require_value("--r-max-mode"));
        } else if (key == "--r-max-factor") {
            args.r_max_factor = std::stod(require_value("--r-max-factor"));
        } else if (key == "--json") {
            args.json = true;
        } else if (key == "--help" || key == "-h") {
            throw std::invalid_argument("");
        } else {
            throw std::invalid_argument("Unknown argument: " + key);
        }
    }

    if (args.method != "wop" && args.method != "wos") {
        throw std::invalid_argument("--method must be one of: wop, wos");
    }
    if (args.n <= 0) {
        throw std::invalid_argument("--n must be positive.");
    }
    if (args.max_steps <= 0) {
        throw std::invalid_argument("--max-steps must be positive.");
    }
    if (args.delta <= 0.0) {
        throw std::invalid_argument("--delta must be positive.");
    }
    if (args.rho_scale <= 0.0) {
        throw std::invalid_argument("--rho-scale must be positive.");
    }
    if (args.rho1_scale <= 1.0) {
        throw std::invalid_argument("--rho1-scale must be greater than 1.");
    }
    if (args.r_max_factor <= 1.0) {
        throw std::invalid_argument("--r-max-factor must be greater than 1.");
    }

    return args;
}

void print_usage() {
    std::cout
        << "Usage: wop_wos_compare_cli --method wop|wos [--x0 \"1.5 0.5 0.5\"] [--n 1000]\n"
        << "       [--seed 12345] [--max-steps 1000000] [--json]\n"
        << "       [--delta 1e-3] [--rho-scale 1.0] [--rho1-scale 2.0]\n"
        << "       [--r-max 1e6] [--r-max-mode escape|project] [--r-max-factor 3.0]\n";
}

int run(const CliArgs& args) {
    const wop::math::Vec3 box_min{0.0, 0.0, 0.0};
    const wop::math::Vec3 box_max{1.0, 1.0, 1.0};
    const wop::math::Vec3 interior{0.5, 0.5, 0.5};
    const wop::math::Vec3 a{0.25, 0.35, 0.45};

    const auto exact_u = [&](const wop::math::Vec3& x) {
        return 1.0 / wop::math::norm(x - a);
    };
    const auto boundary_f = [&](const wop::math::Vec3& y, std::optional<int>) {
        return exact_u(y);
    };

    wop::rng::Rng rng(args.seed);
    wop::estimation::EstimateResult result{};

    if (args.method == "wop") {
        auto poly = wop::geometry::make_axis_aligned_box(box_min, box_max);
        poly = wop::geometry::orient_normals(poly, interior);
        result = wop::solver::estimate_wop(
            poly,
            args.x0,
            boundary_f,
            args.n,
            rng,
            std::nullopt,
            std::nullopt,
            1e-14,
            args.max_steps,
            0.0,
            args.r_max,
            args.r_max_mode,
            args.r_max_factor);
    } else {
        result = wop::solver::estimate_wos_box(
            args.x0,
            box_min,
            box_max,
            boundary_f,
            args.n,
            rng,
            args.delta,
            args.rho_scale,
            args.rho1_scale,
            args.max_steps,
            0.0);
    }

    const double exact = exact_u(args.x0);
    const double abs_error = std::abs(result.J - exact);

    if (args.json) {
        std::cout << std::setprecision(17)
                  << "{"
                  << "\"method\":\"" << args.method << "\"," 
                  << "\"x0\":[" << args.x0.x << "," << args.x0.y << "," << args.x0.z << "],"
                  << "\"n_total\":" << result.n_total << ","
                  << "\"n_truncated\":" << result.n_truncated << ","
                  << "\"J\":" << result.J << ","
                  << "\"exact\":" << exact << ","
                  << "\"abs_error\":" << abs_error << ","
                  << "\"S2\":" << result.S2 << ","
                  << "\"eps\":" << result.eps << ","
                  << "\"mean_steps\":" << result.mean_steps
                  << "}\n";
        return 0;
    }

    std::cout << "method: " << args.method << "\n";
    std::cout << "J: " << result.J << "\n";
    std::cout << "exact: " << exact << "\n";
    std::cout << "abs_error: " << abs_error << "\n";
    std::cout << "eps: " << result.eps << "\n";
    std::cout << "mean_steps: " << result.mean_steps << "\n";
    return 0;
}

}  // namespace

int main(int argc, char** argv) {
    try {
        const CliArgs args = parse_args(argc, argv);
        return run(args);
    } catch (const std::invalid_argument& ex) {
        if (std::string(ex.what()).empty()) {
            print_usage();
            return 0;
        }
        std::cerr << "Error: " << ex.what() << "\n";
        print_usage();
        return 2;
    } catch (const std::exception& ex) {
        std::cerr << "Fatal: " << ex.what() << "\n";
        return 1;
    }
}
''', encoding="utf-8")
runner_cmake.write_text(r'''
cmake_minimum_required(VERSION 3.20)
project(wop_wos_compare_cli LANGUAGES CXX)

set(CMAKE_CXX_STANDARD 20)
set(CMAKE_CXX_STANDARD_REQUIRED ON)
set(CMAKE_CXX_EXTENSIONS OFF)

if(NOT DEFINED WOP_CPP_ROOT)
    message(FATAL_ERROR "WOP_CPP_ROOT must be provided via -DWOP_CPP_ROOT=...")
endif()

set(WOP_BUILD_TESTS OFF CACHE BOOL "" FORCE)
set(WOP_ENABLE_RELEASE_IPO OFF CACHE BOOL "" FORCE)
add_subdirectory(${WOP_CPP_ROOT} wop_cpp_build)

add_executable(wop_wos_compare_cli main.cpp)
target_link_libraries(wop_wos_compare_cli PRIVATE wop_core)
''', encoding="utf-8")

print("Prepared files:")
print(runner_cpp)
print(runner_cmake)
            


Prepared files:
c:\Users\Master\Desktop\Data\Диплом\tmp\jupyter-notebook\wop-vs-wos-cpp-unit-cube\main.cpp
c:\Users\Master\Desktop\Data\Диплом\tmp\jupyter-notebook\wop-vs-wos-cpp-unit-cube\CMakeLists.txt


In [3]:
build_dir = TMP_ROOT / "build"

subprocess.run(
    [
        "cmake",
        "-S", str(TMP_ROOT),
        "-B", str(build_dir),
        f"-DWOP_CPP_ROOT={CPP_ROOT}",
    ],
    check=True,
)

subprocess.run(
    [
        "cmake",
        "--build", str(build_dir),
        "--config", "Release",
        "--target", "wop_wos_compare_cli",
    ],
    check=True,
)

runner_candidates = [
    build_dir / "wop_wos_compare_cli",
    build_dir / "wop_wos_compare_cli.exe",
    build_dir / "Release" / "wop_wos_compare_cli.exe",
]

RUNNER = next((p for p in runner_candidates if p.exists()), None)
if RUNNER is None:
    raise FileNotFoundError("Could not find built runner executable")

print("RUNNER =", RUNNER)
            


RUNNER = c:\Users\Master\Desktop\Data\Диплом\tmp\jupyter-notebook\wop-vs-wos-cpp-unit-cube\build\Release\wop_wos_compare_cli.exe


In [4]:
def run_cpp(method: str, n_paths: int, seed: int) -> dict[str, float]:
    cmd = [
        str(RUNNER),
        "--method", method,
        "--x0", f"{X0[0]} {X0[1]} {X0[2]}",
        "--n", str(int(n_paths)),
        "--seed", str(int(seed)),
        "--max-steps", str(int(MAX_STEPS)),
        "--json",
    ]

    if method == "wop":
        cmd.extend([
            "--r-max", str(float(R_MAX)),
            "--r-max-mode", R_MAX_MODE,
            "--r-max-factor", str(float(R_MAX_FACTOR)),
        ])
    elif method == "wos":
        cmd.extend([
            "--delta", str(float(DELTA)),
            "--rho-scale", str(float(RHO_SCALE)),
            "--rho1-scale", str(float(RHO1_SCALE)),
        ])
    else:
        raise ValueError(f"Unknown method: {method}")

    t0 = time.perf_counter()
    out = subprocess.check_output(cmd, cwd=CPP_ROOT, text=True)
    elapsed = time.perf_counter() - t0

    row = json.loads(out)
    row["wall_time_sec"] = elapsed
    return row
            


In [ ]:
rows: list[dict[str, float]] = []

for n in N_VALUES:
    for method_index, method in enumerate(METHODS):
        seed = SEED_BASE + method_index * 1000 + int(math.log10(n))
        row = run_cpp(method=method, n_paths=n, seed=seed)
        rows.append(row)
        print(
            f"method={method:>3} n={n:>7} J={row['J']:.8f} "
            f"abs_error={row['abs_error']:.3e} time={row['wall_time_sec']:.2f}s"
        )

results_path = TMP_ROOT / "results.json"
results_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
print("Saved raw results to", results_path)
            


method=wop n=    100 J=0.90495085 abs_error=1.113e-01 time=7.03s
method=wos n=    100 J=0.80920400 abs_error=1.553e-02 time=0.09s
method=wop n=   1000 J=0.78096286 abs_error=1.271e-02 time=86.48s
method=wos n=   1000 J=0.79066148 abs_error=3.014e-03 time=0.53s
method=wop n=  10000 J=0.78744757 abs_error=6.228e-03 time=5627.56s
method=wos n=  10000 J=0.79272507 abs_error=9.507e-04 time=5.85s


In [ ]:
rows_by_key = {(row["method"], int(row["n_total"])): row for row in rows}

table_lines = [
    "| method | n | J | abs_error | eps | mean_steps | time_s |",
    "|---|---:|---:|---:|---:|---:|---:|",
]

for method in METHODS:
    for n in N_VALUES:
        row = rows_by_key[(method, n)]
        table_lines.append(
            f"| {method} | {n} | {row['J']:.8f} | {row['abs_error']:.3e} | "
            f"{row['eps']:.3e} | {row['mean_steps']:.2f} | {row['wall_time_sec']:.2f} |"
        )

print("
".join(table_lines))

summary = []
for n in N_VALUES:
    wop_row = rows_by_key[("wop", n)]
    wos_row = rows_by_key[("wos", n)]

    err_ratio = (wos_row["abs_error"] / wop_row["abs_error"]) if wop_row["abs_error"] > 0.0 else float("nan")
    time_ratio = (wos_row["wall_time_sec"] / wop_row["wall_time_sec"]) if wop_row["wall_time_sec"] > 0.0 else float("nan")

    summary.append({
        "n": n,
        "wop_abs_error": wop_row["abs_error"],
        "wos_abs_error": wos_row["abs_error"],
        "abs_error_ratio_wos_over_wop": err_ratio,
        "wop_time_s": wop_row["wall_time_sec"],
        "wos_time_s": wos_row["wall_time_sec"],
        "time_ratio_wos_over_wop": time_ratio,
    })

pprint(summary)
            
